# 07 - Refinando o dataset: colunas derivadas do alvo

Esta etapa le `csvs/03_dataset_limpo_sem_corridas_encurtadas.csv` e remove as duplicatas do alvo apontadas na secao 2 do `06_Avaliacao_Colunas.ipynb`. O arquivo de entrada nao e alterado; o resultado sai em `csvs/07_dataset_refinado.csv`.

O objetivo e prever a ordem final de cada corrida. O unico alvo e `rd_position_display_order`: vai de 1 (vencedor) a N e coloca os nao classificados (DNF, DNS, DSQ...) no fim, na ordem oficial.

| Coluna | Decisao |
|---|---|
| `rd_position_display_order` | Alvo. |
| `rd_position_number` | Removida: mesma ordem, restrita aos classificados. |
| `rd_position_text` | Removida: mesma ordem em texto, com codigos no lugar dos nao classificados. |
| `rd_race_points` | Removida: deriva da posicao final. |

Historicos defasados (abandonos e pontos anteriores) devem ser calculados a partir do arquivo 03, que continua com essas colunas.

In [1]:
import pandas as pd
from IPython.display import display

ENTRADA = 'csvs/03_dataset_limpo_sem_corridas_encurtadas.csv'
SAIDA = 'csvs/07_dataset_refinado.csv'
ALVO = 'rd_position_display_order'

df = pd.read_csv(ENTRADA, low_memory=False)
print(f'Linhas: {len(df):,} | Colunas: {df.shape[1]} | Corridas: {df.race_id.nunique()}')

Linhas: 8,462 | Colunas: 137 | Corridas: 403


## 1. Conferencias

Antes de remover, confirmamos que as tres colunas nao trazem nada alem do alvo:

- o alvo nao tem nulos e e uma permutacao de 1..N em cada corrida;
- `rd_position_number` e igual ao alvo em todas as linhas classificadas;
- `rd_position_text` e numerica exatamente nessas linhas e tem um codigo nas demais;
- `rd_race_points` so aparece nas primeiras posicoes da ordem.

In [2]:
assert df[ALVO].notna().all()
por_corrida = df.groupby('race_id')[ALVO].agg(['size', 'min', 'max', 'nunique'])
assert (por_corrida['min'].eq(1) & por_corrida['max'].eq(por_corrida['size'])
        & por_corrida['nunique'].eq(por_corrida['size'])).all()

classificado = df['rd_position_number'].notna()
assert df.loc[classificado, ALVO].eq(df.loc[classificado, 'rd_position_number']).all()
assert pd.to_numeric(df['rd_position_text'], errors='coerce').notna().eq(classificado).all()

display(df.loc[~classificado, 'rd_position_text'].value_counts().rename('linhas').to_frame())
pontuou = df['rd_race_points'].notna().map({True: 'com pontos', False: 'sem pontos'})
display(df.groupby(pontuou)[ALVO].agg(['count', 'min', 'max']))

,linhas
rd_position_text,
DNF,1462
DNS,56
DSQ,28
NC,14
DNQ,2
DNP,2


,count,min,max
rd_race_points,,,
com pontos,3800,1,10
sem pontos,4662,7,24


## 2. Remocao

In [3]:
REMOVIDAS = ['rd_position_number', 'rd_position_text', 'rd_race_points']

df_refinado = df.drop(columns=REMOVIDAS)

assert len(df_refinado) == len(df)
assert ALVO in df_refinado.columns

df_refinado.to_csv(SAIDA, index=False)
print(f'Colunas: {df.shape[1]} -> {df_refinado.shape[1]} (removidas: {", ".join(REMOVIDAS)})')
print(f'Alvo: {ALVO}')
print(f'Arquivo salvo: {SAIDA}')

Colunas: 137 -> 134 (removidas: rd_position_number, rd_position_text, rd_race_points)
Alvo: rd_position_display_order
Arquivo salvo: csvs/07_dataset_refinado.csv


## 3. Historicos defasados: resultado da propria corrida

A secao 3 do `06_Avaliacao_Colunas.ipynb` classifica `rd_race_laps`, `rd_race_positions_gained`, `rd_race_pit_stops`, `rd_race_fastest_lap`, `rd_race_driver_of_the_day`, `rd_race_grand_slam` e `sess_fastest_lap_time_millis` como vazamento: so existem depois da largada. Parte dessa informacao tem valor como historico defasado (`shift(1)`), calculado a partir de `df` (arquivo 03, que ainda tem essas colunas):

| Coluna | Decisao |
|---|---|
| `rd_position_text` / `rd_race_laps` | Vira **taxa de abandono defasada**, por piloto e por equipe. Nao classificados completam em media 38% das voltas e vao para o fim da ordem (secao 3 de `06`): confiabilidade e um sinal que falta hoje nas features. |
| `rd_race_positions_gained` | Vira **media defasada de posicoes ganhas** do piloto (capacidade de ultrapassar). Em `06` a coluna bruta e sempre `grid - posicao final`: contem o alvo. |
| `rd_race_pit_stops`, `rd_race_fastest_lap`, `sess_fastest_lap_time_millis`, `rd_race_driver_of_the_day`, `rd_race_grand_slam` | Descartadas sem substituto. Ganho de uma versao defasada seria pequeno, e piloto do dia so existe a partir de 2016. |

O historico usa toda a carreira do piloto/equipe ate a corrida anterior (`shift(1).expanding(min_periods=1).mean()`), no mesmo estilo de `constructor_prior_avg_finish` em `feature_engineering.py`.</cell_type>


In [4]:
historico = df[[
    'race_id', 'race_date', 'race_year', 'race_round', 'driver_id', 'constructor_id',
    'rd_race_positions_gained',
]].copy()
historico['abandonou'] = (~classificado).astype(int)
historico['race_date'] = pd.to_datetime(historico['race_date'])
historico = historico.sort_values(
    ['race_date', 'race_year', 'race_round', 'race_id', 'driver_id'], kind='mergesort'
).reset_index(drop=True)

piloto = historico.groupby('driver_id', sort=False)
historico['driver_prior_dnf_rate'] = piloto['abandonou'].transform(
    lambda serie: serie.shift(1).expanding(min_periods=1).mean()
)
historico['driver_prior_positions_gained_avg'] = piloto['rd_race_positions_gained'].transform(
    lambda serie: serie.shift(1).expanding(min_periods=1).mean()
)

# Resume por equipe x corrida antes do historico, para nao vazar o resultado
# do companheiro na propria etapa (mesmo padrao de constructor_prior_avg_finish
# em feature_engineering.py).
equipe_corrida = (
    historico.groupby(['race_id', 'constructor_id'], as_index=False, sort=False)
    .agg(
        race_date=('race_date', 'first'), race_year=('race_year', 'first'),
        race_round=('race_round', 'first'), team_dnf_rate=('abandonou', 'mean'),
    )
    .sort_values(['race_date', 'race_year', 'race_round', 'race_id', 'constructor_id'], kind='mergesort')
    .reset_index(drop=True)
)
equipe_corrida['constructor_prior_dnf_rate'] = equipe_corrida.groupby('constructor_id', sort=False)['team_dnf_rate'].transform(
    lambda serie: serie.shift(1).expanding(min_periods=1).mean()
)

historico = historico.merge(
    equipe_corrida[['race_id', 'constructor_id', 'constructor_prior_dnf_rate']],
    on=['race_id', 'constructor_id'], how='left', validate='many_to_one',
)

# so a primeira corrida de cada piloto/equipe fica sem historico de abandono
assert historico['driver_prior_dnf_rate'].isna().sum() == historico['driver_id'].nunique()
assert equipe_corrida['constructor_prior_dnf_rate'].isna().sum() == equipe_corrida['constructor_id'].nunique()

display(historico.head())

,race_id,race_date,race_year,race_round,driver_id,constructor_id,rd_race_positions_gained,abandonou,driver_prior_dnf_rate,driver_prior_positions_gained_avg,constructor_prior_dnf_rate
0,698,2003-03-09,2003,1,antonio-pizzonia,jaguar,5.0,0,NaN,NaN,NaN
1,698,2003-03-09,2003,1,cristiano-da-matta,toyota,NaN,1,NaN,NaN,NaN
2,698,2003-03-09,2003,1,david-coulthard,mclaren,10.0,0,NaN,NaN,NaN
3,698,2003-03-09,2003,1,fernando-alonso,renault,3.0,0,NaN,NaN,NaN
4,698,2003-03-09,2003,1,giancarlo-fisichella,jordan,1.0,0,NaN,NaN,NaN


In [5]:
NOVAS = ['driver_prior_dnf_rate', 'constructor_prior_dnf_rate', 'driver_prior_positions_gained_avg']
DESCARTADAS = [
    'rd_race_laps', 'rd_race_positions_gained', 'rd_race_pit_stops', 'rd_race_fastest_lap',
    'rd_race_driver_of_the_day', 'rd_race_grand_slam', 'sess_fastest_lap_time_millis',
]

antes = df_refinado.shape[1]
df_refinado = df_refinado.merge(
    historico[['race_id', 'driver_id'] + NOVAS],
    on=['race_id', 'driver_id'], how='left', validate='one_to_one',
)
df_refinado = df_refinado.drop(columns=DESCARTADAS)

assert len(df_refinado) == len(df)
assert set(NOVAS) <= set(df_refinado.columns)
assert not set(DESCARTADAS) & set(df_refinado.columns)

df_refinado.to_csv(SAIDA, index=False)
print(f'Colunas: {antes} -> {df_refinado.shape[1]} (descartadas: {len(DESCARTADAS)}, novas: {len(NOVAS)})')
print(f'Cobertura (nao nulo): {df_refinado[NOVAS].notna().mean().mul(100).round(1).to_dict()}')
print(f'Arquivo salvo: {SAIDA}')

Colunas: 134 -> 130 (descartadas: 7, novas: 3)
Cobertura (nao nulo): {'driver_prior_dnf_rate': 98.6, 'constructor_prior_dnf_rate': 99.1, 'driver_prior_positions_gained_avg': 98.1}
Arquivo salvo: csvs/07_dataset_refinado.csv


## 4. Historicos defasados: standings

A secao 4 de `06_Avaliacao_Colunas.ipynb` classifica `dstand_*`, `cstand_*`, `race_drivers_championship_decider` e `race_constructors_championship_decider` como vazamento: sao a classificacao do campeonato ja atualizada com a propria corrida. A melhor feature que da para tirar dai e a posicao e os pontos do piloto/equipe **na rodada anterior da mesma temporada** (`shift(1)` agrupando por piloto/equipe e `race_year`, ordenado por `race_round`). Na primeira rodada de cada temporada (ou na estreia do piloto/equipe naquela temporada) nao ha rodada anterior: pontos ficam 0 e posicao fica vazia (`NaN`).

| Coluna | Decisao |
|---|---|
| `dstand_position_number`, `dstand_points` | Viram `driver_prior_standing_position` e `driver_prior_standing_points`, defasadas por temporada. |
| `cstand_position_number`, `cstand_points` | Viram `constructor_prior_standing_position` e `constructor_prior_standing_points`, defasadas por temporada. |
| `dstand_positions_gained`, `dstand_championship_won`, `cstand_position_text`, `cstand_engine_manufacturer_id`, `cstand_positions_gained`, `cstand_championship_won`, `race_drivers_championship_decider`, `race_constructors_championship_decider` | Descartadas sem substituto. |</cell_type>


In [6]:
standings = df[[
    'race_id', 'race_date', 'race_year', 'race_round', 'driver_id', 'constructor_id',
    'dstand_position_number', 'dstand_points', 'cstand_position_number', 'cstand_points',
]].copy()
standings['race_date'] = pd.to_datetime(standings['race_date'])
standings = standings.sort_values(
    ['race_date', 'race_year', 'race_round', 'race_id', 'driver_id'], kind='mergesort'
).reset_index(drop=True)

# cstand_* e a mesma classificacao para os dois companheiros de equipe na mesma corrida.
max_por_equipe = standings.groupby(['race_id', 'constructor_id'])[
    ['cstand_position_number', 'cstand_points']
].nunique(dropna=False).to_numpy().max()
assert max_por_equipe <= 1

piloto = standings.groupby(['driver_id', 'race_year'], sort=False)
standings['driver_prior_standing_position'] = piloto['dstand_position_number'].transform(lambda serie: serie.shift(1))
standings['driver_prior_standing_points'] = piloto['dstand_points'].transform(lambda serie: serie.shift(1).fillna(0))

equipe_standing = (
    standings.groupby(['race_id', 'constructor_id'], as_index=False, sort=False)
    .agg(
        race_date=('race_date', 'first'), race_year=('race_year', 'first'), race_round=('race_round', 'first'),
        cstand_position_number=('cstand_position_number', 'first'), cstand_points=('cstand_points', 'first'),
    )
    .sort_values(['race_date', 'race_year', 'race_round', 'race_id', 'constructor_id'], kind='mergesort')
    .reset_index(drop=True)
)
equipe = equipe_standing.groupby(['constructor_id', 'race_year'], sort=False)
equipe_standing['constructor_prior_standing_position'] = equipe['cstand_position_number'].transform(lambda serie: serie.shift(1))
equipe_standing['constructor_prior_standing_points'] = equipe['cstand_points'].transform(lambda serie: serie.shift(1).fillna(0))

standings = standings.merge(
    equipe_standing[['race_id', 'constructor_id', 'constructor_prior_standing_position', 'constructor_prior_standing_points']],
    on=['race_id', 'constructor_id'], how='left', validate='many_to_one',
)

# rodada 1 (ou estreia do piloto/equipe na temporada): pontos = 0, posicao vazia
primeira_rodada = standings['race_round'].eq(1)
assert standings.loc[primeira_rodada, 'driver_prior_standing_points'].eq(0).all()
assert standings.loc[primeira_rodada, 'driver_prior_standing_position'].isna().all()
assert equipe_standing.loc[equipe_standing['race_round'].eq(1), 'constructor_prior_standing_points'].eq(0).all()
assert equipe_standing.loc[equipe_standing['race_round'].eq(1), 'constructor_prior_standing_position'].isna().all()

display(standings.head())

,race_id,race_date,race_year,race_round,driver_id,constructor_id,dstand_position_number,dstand_points,cstand_position_number,cstand_points,driver_prior_standing_position,driver_prior_standing_points,constructor_prior_standing_position,constructor_prior_standing_points
0,698,2003-03-09,2003,1,antonio-pizzonia,jaguar,13.0,0.0,9.0,0.0,NaN,0.0,NaN,0.0
1,698,2003-03-09,2003,1,cristiano-da-matta,toyota,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0
2,698,2003-03-09,2003,1,david-coulthard,mclaren,1.0,10.0,1.0,16.0,NaN,0.0,NaN,0.0
3,698,2003-03-09,2003,1,fernando-alonso,renault,7.0,2.0,3.0,6.0,NaN,0.0,NaN,0.0
4,698,2003-03-09,2003,1,giancarlo-fisichella,jordan,12.0,0.0,8.0,0.0,NaN,0.0,NaN,0.0


In [7]:
NOVAS_STANDING = [
    'driver_prior_standing_position', 'driver_prior_standing_points',
    'constructor_prior_standing_position', 'constructor_prior_standing_points',
]
DESCARTADAS_STANDING = [
    'dstand_position_number', 'dstand_points', 'dstand_positions_gained', 'dstand_championship_won',
    'cstand_position_number', 'cstand_position_text', 'cstand_engine_manufacturer_id',
    'cstand_points', 'cstand_positions_gained', 'cstand_championship_won',
    'race_drivers_championship_decider', 'race_constructors_championship_decider',
]

antes = df_refinado.shape[1]
df_refinado = df_refinado.merge(
    standings[['race_id', 'driver_id'] + NOVAS_STANDING],
    on=['race_id', 'driver_id'], how='left', validate='one_to_one',
)
df_refinado = df_refinado.drop(columns=DESCARTADAS_STANDING)

assert len(df_refinado) == len(df)
assert set(NOVAS_STANDING) <= set(df_refinado.columns)
assert not set(DESCARTADAS_STANDING) & set(df_refinado.columns)

df_refinado.to_csv(SAIDA, index=False)
print(f'Colunas: {antes} -> {df_refinado.shape[1]} (descartadas: {len(DESCARTADAS_STANDING)}, novas: {len(NOVAS_STANDING)})')
print(f'Cobertura (nao nulo): {df_refinado[NOVAS_STANDING].notna().mean().mul(100).round(1).to_dict()}')
print(f'Arquivo salvo: {SAIDA}')

Colunas: 130 -> 122 (descartadas: 12, novas: 4)
Cobertura (nao nulo): {'driver_prior_standing_position': 91.4, 'driver_prior_standing_points': 100.0, 'constructor_prior_standing_position': 93.1, 'constructor_prior_standing_points': 100.0}
Arquivo salvo: csvs/07_dataset_refinado.csv


## 5. Resumos de carreira atuais: descarte

A secao 5 de `06_Avaliacao_Colunas.ipynb` classifica 53 colunas `driver_*`, `constructor_*`, `engmfr_*` e `tyremfr_*` com sufixo `_total_*`/`_best_*` (mais `circuit_total_races_held` e `gp_total_races_held`) como vazamento: sao fotografias de hoje, que ja incluem resultados futuros nas corridas mais antigas de cada piloto/equipe.

Os substitutos defasados de carreira do piloto e da equipe (entradas, vitorias, taxa de vitoria, podios, media de posicao/pontos recentes) ja existem, com o mesmo padrao `shift(1)`, em `feature_engineering.py` (`driver_prior_*` / `constructor_prior_*`) — fora do escopo desta celula.

Faltam dois que valem a pena acrescentar aqui, tambem com `shift(1)`:

| Feature | Como |
|---|---|
| `constructor_prior_poles`, `constructor_prior_podiums` | Poles e podios acumulados da equipe antes da corrida atual (soma dos dois carros, desde a estreia). |
| `engmfr_prior_avg_finish_last5` | Desempenho recente do fabricante do motor: media da posicao final das equipes que usam esse motor, nas ultimas 5 corridas (mesma janela de `driver_prior_avg_finish_last5` em `feature_engineering.py`). |</cell_type>


In [8]:
N = 5

carreira = df[[
    'race_id', 'race_date', 'race_year', 'race_round', 'driver_id', 'constructor_id', 'engmfr_id',
    'rd_race_pole_position',
]].copy()
carreira['race_date'] = pd.to_datetime(carreira['race_date'])
carreira['podio'] = df[ALVO].le(3).astype(int)
carreira['posicao_final'] = df[ALVO]
carreira = carreira.sort_values(
    ['race_date', 'race_year', 'race_round', 'race_id', 'driver_id'], kind='mergesort'
).reset_index(drop=True)

# uma equipe usa um unico motor por corrida
max_motores_por_equipe = carreira.groupby(['race_id', 'constructor_id'])['engmfr_id'].nunique().max()
assert max_motores_por_equipe == 1

# equipe x corrida: poles e podios somados entre os companheiros, motor da equipe naquela etapa
equipe_carreira = (
    carreira.groupby(['race_id', 'constructor_id'], as_index=False, sort=False)
    .agg(
        race_date=('race_date', 'first'), race_year=('race_year', 'first'), race_round=('race_round', 'first'),
        engmfr_id=('engmfr_id', 'first'),
        team_poles=('rd_race_pole_position', 'sum'),
        team_podiums=('podio', 'sum'),
        team_avg_finish=('posicao_final', 'mean'),
    )
    .sort_values(['race_date', 'race_year', 'race_round', 'race_id', 'constructor_id'], kind='mergesort')
    .reset_index(drop=True)
)
equipe = equipe_carreira.groupby('constructor_id', sort=False)
equipe_carreira['constructor_prior_poles'] = equipe['team_poles'].transform(
    lambda serie: serie.shift(1).fillna(0).cumsum()
).astype('int64')
equipe_carreira['constructor_prior_podiums'] = equipe['team_podiums'].transform(
    lambda serie: serie.shift(1).fillna(0).cumsum()
).astype('int64')

# motor x corrida: media entre as equipes que usam aquele motor na corrida
motor_corrida = (
    equipe_carreira.groupby(['race_id', 'engmfr_id'], as_index=False, sort=False)
    .agg(
        race_date=('race_date', 'first'), race_year=('race_year', 'first'), race_round=('race_round', 'first'),
        engmfr_avg_finish=('team_avg_finish', 'mean'),
    )
    .sort_values(['race_date', 'race_year', 'race_round', 'race_id', 'engmfr_id'], kind='mergesort')
    .reset_index(drop=True)
)
motor = motor_corrida.groupby('engmfr_id', sort=False)
motor_corrida['engmfr_prior_avg_finish_last5'] = motor['engmfr_avg_finish'].transform(
    lambda serie: serie.shift(1).rolling(window=N, min_periods=1).mean()
)

equipe_carreira = equipe_carreira.merge(
    motor_corrida[['race_id', 'engmfr_id', 'engmfr_prior_avg_finish_last5']],
    on=['race_id', 'engmfr_id'], how='left', validate='many_to_one',
)

carreira = carreira.merge(
    equipe_carreira[[
        'race_id', 'constructor_id', 'constructor_prior_poles', 'constructor_prior_podiums',
        'engmfr_prior_avg_finish_last5',
    ]],
    on=['race_id', 'constructor_id'], how='left', validate='many_to_one',
)

# estreia da equipe: zero poles/podios acumulados
estreia_equipe = equipe_carreira.groupby('constructor_id', sort=False).cumcount().eq(0)
assert equipe_carreira.loc[estreia_equipe, 'constructor_prior_poles'].eq(0).all()
assert equipe_carreira.loc[estreia_equipe, 'constructor_prior_podiums'].eq(0).all()

display(carreira.head())

,race_id,race_date,race_year,race_round,driver_id,constructor_id,engmfr_id,rd_race_pole_position,podio,posicao_final,constructor_prior_poles,constructor_prior_podiums,engmfr_prior_avg_finish_last5
0,698,2003-03-09,2003,1,antonio-pizzonia,jaguar,cosworth,0,0,13,0,0,NaN
1,698,2003-03-09,2003,1,cristiano-da-matta,toyota,toyota,0,0,18,0,0,NaN
2,698,2003-03-09,2003,1,david-coulthard,mclaren,mercedes,0,1,1,0,0,NaN
3,698,2003-03-09,2003,1,fernando-alonso,renault,renault,0,0,7,0,0,NaN
4,698,2003-03-09,2003,1,giancarlo-fisichella,jordan,ford,0,0,12,0,0,NaN


In [9]:
NOVAS_CARREIRA = ['constructor_prior_poles', 'constructor_prior_podiums', 'engmfr_prior_avg_finish_last5']
DESCARTADAS_CARREIRA = [
    # driver_*
    'driver_best_championship_position', 'driver_best_starting_grid_position', 'driver_best_race_result',
    'driver_total_championship_wins', 'driver_total_race_entries', 'driver_total_race_starts',
    'driver_total_race_wins', 'driver_total_race_laps', 'driver_total_podiums', 'driver_total_points',
    'driver_total_pole_positions', 'driver_total_fastest_laps', 'driver_total_driver_of_the_day',
    'driver_total_grand_slams',
    # constructor_*
    'constructor_best_championship_position', 'constructor_best_starting_grid_position',
    'constructor_best_race_result', 'constructor_total_championship_wins', 'constructor_total_race_entries',
    'constructor_total_race_starts', 'constructor_total_race_wins', 'constructor_total_1_and_2_finishes',
    'constructor_total_race_laps', 'constructor_total_podiums', 'constructor_total_podium_races',
    'constructor_total_points', 'constructor_total_championship_points', 'constructor_total_pole_positions',
    'constructor_total_fastest_laps',
    # engmfr_*
    'engmfr_best_championship_position', 'engmfr_best_starting_grid_position', 'engmfr_best_race_result',
    'engmfr_total_championship_wins', 'engmfr_total_race_entries', 'engmfr_total_race_starts',
    'engmfr_total_race_wins', 'engmfr_total_race_laps', 'engmfr_total_podiums', 'engmfr_total_podium_races',
    'engmfr_total_points', 'engmfr_total_championship_points', 'engmfr_total_pole_positions',
    'engmfr_total_fastest_laps',
    # tyremfr_*
    'tyremfr_total_race_entries', 'tyremfr_total_race_starts', 'tyremfr_total_race_wins',
    'tyremfr_total_race_laps', 'tyremfr_total_podiums', 'tyremfr_total_podium_races',
    'tyremfr_total_pole_positions', 'tyremfr_total_fastest_laps',
    # outros resumos atuais
    'circuit_total_races_held', 'gp_total_races_held',
]
assert len(DESCARTADAS_CARREIRA) == 53

antes = df_refinado.shape[1]
df_refinado = df_refinado.merge(
    carreira[['race_id', 'driver_id'] + NOVAS_CARREIRA],
    on=['race_id', 'driver_id'], how='left', validate='one_to_one',
)
df_refinado = df_refinado.drop(columns=DESCARTADAS_CARREIRA)

assert len(df_refinado) == len(df)
assert set(NOVAS_CARREIRA) <= set(df_refinado.columns)
assert not set(DESCARTADAS_CARREIRA) & set(df_refinado.columns)

df_refinado.to_csv(SAIDA, index=False)
print(f'Colunas: {antes} -> {df_refinado.shape[1]} (descartadas: {len(DESCARTADAS_CARREIRA)}, novas: {len(NOVAS_CARREIRA)})')
print(f'Cobertura (nao nulo): {df_refinado[NOVAS_CARREIRA].notna().mean().mul(100).round(1).to_dict()}')
print(f'Arquivo salvo: {SAIDA}')

Colunas: 122 -> 72 (descartadas: 53, novas: 3)
Cobertura (nao nulo): {'constructor_prior_poles': 100.0, 'constructor_prior_podiums': 100.0, 'engmfr_prior_avg_finish_last5': 99.5}
Arquivo salvo: csvs/07_dataset_refinado.csv


## 6. Agregados da temporada: descarte

A secao 6 de `06_Avaliacao_Colunas.ipynb` classifica `ed_entrant_ids`, `ed_entrant_names`, `ed_rounds_text`, `ed_test_driver`, `chassis_ids`, `chassis_names`, `engine_ids` e `engine_names` como vazamento sutil: sao agregados da temporada inteira, entao revelam pecas/substituicoes que so estreiam depois da corrida atual.

Sem substituto: `chassis_ids` equivale a equipe x temporada (253 = 253 combinacoes, ja conferido em `06`), mas o chassi de 2025/2026 nunca aparece no treino. O desempenho da equipe na temporada ja e capturado por `constructor_prior_avg_*_last5` (`feature_engineering.py`) e pelo standing anterior (secao 4 deste notebook), sem esse problema. `ed_entrant_*` repete a equipe (`constructor_id`/`constructor_name`, ja redundante). `ed_rounds_text` e `ed_test_driver` nao tem uso legitimo pre-largada.</cell_type>


In [10]:
DESCARTADAS_TEMPORADA = [
    'ed_entrant_ids', 'ed_entrant_names', 'ed_rounds_text', 'ed_test_driver',
    'chassis_ids', 'chassis_names', 'engine_ids', 'engine_names',
]
assert len(DESCARTADAS_TEMPORADA) == 8

antes = df_refinado.shape[1]
df_refinado = df_refinado.drop(columns=DESCARTADAS_TEMPORADA)

assert len(df_refinado) == len(df)
assert not set(DESCARTADAS_TEMPORADA) & set(df_refinado.columns)

df_refinado.to_csv(SAIDA, index=False)
print(f'Colunas: {antes} -> {df_refinado.shape[1]} (descartadas: {len(DESCARTADAS_TEMPORADA)})')
print(f'Arquivo salvo: {SAIDA}')

Colunas: 72 -> 64 (descartadas: 8)
Arquivo salvo: csvs/07_dataset_refinado.csv


## 7. Identificadores e textos: metadados, idade e flags

A secao 7 de `06_Avaliacao_Colunas.ipynb` classifica `race_id`, `race_date`, `driver_date_of_birth`, os 5 nomes (`driver_name`, `constructor_name`, `engmfr_name`, `tyremfr_name`, `circuit_name`) e as 3 colunas `*_text` (`rd_race_grid_position_text`, `sess_grid_position_text`, `sess_quali_position_text`) como identificador ou texto redundante.

| Coluna | Decisao |
|---|---|
| `race_id` | Metadado, nao feature: mantido no dataset para servir de `group` num modelo de ranking, na validacao e nas metricas por corrida. |
| `race_date` | Metadado, nao feature: ja foi usado so para ordenar os dados e calcular os historicos (secoes 3 a 5); mantido para eventuais splits temporais. |
| `driver_date_of_birth` | Vira `driver_age_years` (idade do piloto na data da corrida); coluna bruta descartada. |
| `driver_name`, `constructor_name`, `engmfr_name`, `tyremfr_name`, `circuit_name` | Descartados: repetem o id correspondente. |
| `rd_race_grid_position_text`, `sess_grid_position_text` | Viram `largou_do_pit_lane` (codigo `PL` em qualquer uma das duas fontes); texto descartado. |
| `sess_quali_position_text` | Vira `sem_tempo_na_classificacao` (codigos `NC`/`EX`/`DSQ`); texto descartado. |

Sem as flags, o grid e a classificacao numericos ficam vazios justamente nesses casos e o modelo nao sabe se e um largada do pit lane, uma exclusao da classificacao ou so um dado faltante.</cell_type>


In [11]:
METADADOS = ['race_id', 'race_date']  # nao sao features: servem de group/ordenacao, nao de entrada do modelo
print(f'Colunas de metadado (mantidas, fora do conjunto de features): {METADADOS}')

textos = df[['race_id', 'driver_id']].copy()
nascimento = pd.to_datetime(df['driver_date_of_birth'], errors='coerce')
race_date_dt = pd.to_datetime(df['race_date'])
textos['driver_age_years'] = ((race_date_dt - nascimento).dt.days / 365.2425).round(2)
textos['largou_do_pit_lane'] = (
    df['rd_race_grid_position_text'].eq('PL') | df['sess_grid_position_text'].eq('PL')
).astype(int)
textos['sem_tempo_na_classificacao'] = df['sess_quali_position_text'].isin(['NC', 'EX', 'DSQ']).astype(int)

assert textos['driver_age_years'].dropna().between(15, 65).all()

print(f"Largaram do pit lane: {textos['largou_do_pit_lane'].sum()}")
print(f"Sem tempo na classificacao: {textos['sem_tempo_na_classificacao'].sum()}")
display(textos.head())

Colunas de metadado (mantidas, fora do conjunto de features): ['race_id', 'race_date']
Largaram do pit lane: 163
Sem tempo na classificacao: 106


,race_id,driver_id,driver_age_years,largou_do_pit_lane,sem_tempo_na_classificacao
0,698,david-coulthard,31.95,0,0
1,698,juan-pablo-montoya,27.47,0,0
2,698,kimi-raikkonen,23.39,1,0
3,698,michael-schumacher,34.18,0,0
4,698,jarno-trulli,28.65,0,0


In [12]:
NOVAS_TEXTO = ['driver_age_years', 'largou_do_pit_lane', 'sem_tempo_na_classificacao']
DESCARTADAS_TEXTO = [
    'driver_date_of_birth',
    'driver_name', 'constructor_name', 'engmfr_name', 'tyremfr_name', 'circuit_name',
    'rd_race_grid_position_text', 'sess_grid_position_text', 'sess_quali_position_text',
]
assert len(DESCARTADAS_TEXTO) == 9

antes = df_refinado.shape[1]
df_refinado = df_refinado.merge(
    textos[['race_id', 'driver_id'] + NOVAS_TEXTO],
    on=['race_id', 'driver_id'], how='left', validate='one_to_one',
)
df_refinado = df_refinado.drop(columns=DESCARTADAS_TEXTO)

assert len(df_refinado) == len(df)
assert set(NOVAS_TEXTO) <= set(df_refinado.columns)
assert not set(DESCARTADAS_TEXTO) & set(df_refinado.columns)
assert set(METADADOS) <= set(df_refinado.columns)

df_refinado.to_csv(SAIDA, index=False)
print(f'Colunas: {antes} -> {df_refinado.shape[1]} (descartadas: {len(DESCARTADAS_TEXTO)}, novas: {len(NOVAS_TEXTO)})')
print(f'Arquivo salvo: {SAIDA}')

Colunas: 64 -> 58 (descartadas: 9, novas: 3)
Arquivo salvo: csvs/07_dataset_refinado.csv


## 8. Medidas redundantes: uma coluna por grupo

A secao 8 de `06_Avaliacao_Colunas.ipynb` classifica 12 colunas como medida redundante: cada uma repete, num formato pior, uma medida que outra coluna ja cobre.

| Grupo | Manter | Descartar |
|---|---|---|
| Grid | `sess_grid_position_number` preenchida com `rd_race_grid_position_number` (`combine_first`) | `rd_race_grid_position_number` |
| Pole | — (ja coberto por `sess_quali_position_number == 1`) | `rd_race_pole_position` |
| Tempo de grid | — | `sess_grid_time_millis` |
| Pista | `race_course_length` e `race_turns` (ja mantidos, constantes dentro da corrida) | `layout_length`, `circuit_length`, `circuit_turns`, `race_laps`, `race_distance` |
| Motor | — (`race_year` ja marca a epoca) | `engine_capacities`, `engine_aspirations`, `engine_configurations` |
| Voltas na classificacao | — (sinal vem da posicao, ja incluida) | `sess_quali_laps` |

Nenhuma coluna nova: so o preenchimento de `sess_grid_position_number` e o descarte das 12 colunas redundantes.</cell_type>


In [13]:
DESCARTADAS_MEDIDAS = [
    'rd_race_grid_position_number', 'rd_race_pole_position', 'sess_grid_time_millis',
    'layout_length', 'circuit_length', 'circuit_turns', 'race_laps', 'race_distance',
    'engine_capacities', 'engine_aspirations', 'engine_configurations', 'sess_quali_laps',
]
assert len(DESCARTADAS_MEDIDAS) == 12

antes = df_refinado.shape[1]
nulos_antes = df_refinado['sess_grid_position_number'].isna().sum()
df_refinado['sess_grid_position_number'] = df_refinado['sess_grid_position_number'].combine_first(
    df_refinado['rd_race_grid_position_number']
)
nulos_depois = df_refinado['sess_grid_position_number'].isna().sum()

df_refinado = df_refinado.drop(columns=DESCARTADAS_MEDIDAS)

assert len(df_refinado) == len(df)
assert not set(DESCARTADAS_MEDIDAS) & set(df_refinado.columns)
assert nulos_depois <= nulos_antes

df_refinado.to_csv(SAIDA, index=False)
print(f'sess_grid_position_number nulos: {nulos_antes} -> {nulos_depois}')
print(f'Colunas: {antes} -> {df_refinado.shape[1]} (descartadas: {len(DESCARTADAS_MEDIDAS)})')
print(f'Arquivo salvo: {SAIDA}')

sess_grid_position_number nulos: 131 -> 130


Colunas: 58 -> 46 (descartadas: 12)
Arquivo salvo: csvs/07_dataset_refinado.csv


## 9. Tempos brutos e sinal fraco

A secao 9 de `06_Avaliacao_Colunas.ipynb` classifica 11 colunas como tempo bruto ou sinal fraco: `sess_fp1_time_millis`, `sess_fp2_time_millis`, `sess_fp3_time_millis`, `sess_quali_time_millis`, `sess_quali_q1_millis`, `sess_quali_q2_millis`, `sess_quali_q3_millis`, `sess_fp1_laps`, `sess_fp2_laps`, `sess_fp3_laps` e `tyremfr_id`.

| Coluna(s) | Decisao |
|---|---|
| `sess_fp1_time_millis`, `sess_fp2_time_millis`, `sess_fp3_time_millis` | Cada uma vira `{sessao}_gap_pct = tempo / melhor_tempo_da_sessao_na_corrida − 1`. Correlacao com o alvo sobe de ~0,06 (bruto) para ~0,50 (relativo). |
| `sess_quali_time_millis`, `sess_quali_q1_millis`, `sess_quali_q2_millis`, `sess_quali_q3_millis` | Juntas viram uma unica `quali_gap_pct`: melhor segmento do piloto (Q3 > Q2 > Q1 > tempo unico de 2003-2005, mesma ordem de `06`) dividido pelo tempo da pole, menos 1. Separados, Q2 e Q3 tem 36% e 59% de nulos por construcao; juntos a cobertura fica quase total. |
| `sess_fp1_laps`, `sess_fp2_laps`, `sess_fp3_laps` | Descartadas: Spearman com o alvo entre -0,03 e 0,06. |
| `tyremfr_id` | Descartada: so varia de 2003 a 2006, sem efeito nos anos de validacao e teste. |

A diferenca de tempo traz algo que a posicao nao traz (chegar em 5o a 0,1s e diferente de chegar em 5o a 1s) e tende a ajudar em modelos de arvore. Vale conferir na validacao se posicao e `gap_pct` da mesma sessao juntas melhoram o resultado.</cell_type>


In [14]:
tempos = df[[
    'race_id', 'driver_id',
    'sess_fp1_time_millis', 'sess_fp2_time_millis', 'sess_fp3_time_millis',
    'sess_quali_time_millis', 'sess_quali_q1_millis', 'sess_quali_q2_millis', 'sess_quali_q3_millis',
]].copy()

for sessao, coluna in [
    ('fp1', 'sess_fp1_time_millis'), ('fp2', 'sess_fp2_time_millis'), ('fp3', 'sess_fp3_time_millis'),
]:
    melhor_da_corrida = tempos.groupby('race_id')[coluna].transform('min')
    tempos[f'{sessao}_gap_pct'] = tempos[coluna] / melhor_da_corrida - 1

# melhor segmento disputado pelo piloto na classificacao: Q3 > Q2 > Q1 > tempo unico (2003-2005)
tempos['quali_melhor_segmento'] = (
    tempos['sess_quali_q3_millis']
    .combine_first(tempos['sess_quali_q2_millis'])
    .combine_first(tempos['sess_quali_q1_millis'])
    .combine_first(tempos['sess_quali_time_millis'])
)
pole_da_corrida = tempos.groupby('race_id')['quali_melhor_segmento'].transform('min')
tempos['quali_gap_pct'] = tempos['quali_melhor_segmento'] / pole_da_corrida - 1

NOVAS_TEMPOS = ['fp1_gap_pct', 'fp2_gap_pct', 'fp3_gap_pct', 'quali_gap_pct']

# ninguem e mais rapido que o melhor/pole da propria corrida
assert (tempos[NOVAS_TEMPOS].dropna() >= -1e-9).all().all()

for coluna in NOVAS_TEMPOS:
    print(f'Cobertura {coluna}: {tempos[coluna].notna().mean():.1%}')
display(tempos[['race_id', 'driver_id'] + NOVAS_TEMPOS].head())

Cobertura fp1_gap_pct: 90.1%
Cobertura fp2_gap_pct: 96.7%
Cobertura fp3_gap_pct: 94.8%
Cobertura quali_gap_pct: 98.3%


,race_id,driver_id,fp1_gap_pct,fp2_gap_pct,fp3_gap_pct,quali_gap_pct
0,698,david-coulthard,0.005537,0.013367,NaN,0.022163
1,698,juan-pablo-montoya,0.016414,0.008881,NaN,0.010645
2,698,kimi-raikkonen,0.000000,0.018820,NaN,0.026350
3,698,michael-schumacher,0.013374,0.006776,NaN,0.000000
4,698,jarno-trulli,0.008982,0.000000,NaN,0.022518


In [15]:
DESCARTADAS_TEMPOS = [
    'sess_fp1_time_millis', 'sess_fp2_time_millis', 'sess_fp3_time_millis',
    'sess_quali_time_millis', 'sess_quali_q1_millis', 'sess_quali_q2_millis', 'sess_quali_q3_millis',
    'sess_fp1_laps', 'sess_fp2_laps', 'sess_fp3_laps',
    'tyremfr_id',
]
assert len(DESCARTADAS_TEMPOS) == 11

antes = df_refinado.shape[1]
df_refinado = df_refinado.merge(
    tempos[['race_id', 'driver_id'] + NOVAS_TEMPOS],
    on=['race_id', 'driver_id'], how='left', validate='one_to_one',
)
df_refinado = df_refinado.drop(columns=DESCARTADAS_TEMPOS)

assert len(df_refinado) == len(df)
assert set(NOVAS_TEMPOS) <= set(df_refinado.columns)
assert not set(DESCARTADAS_TEMPOS) & set(df_refinado.columns)

df_refinado.to_csv(SAIDA, index=False)
print(f'Colunas: {antes} -> {df_refinado.shape[1]} (descartadas: {len(DESCARTADAS_TEMPOS)}, novas: {len(NOVAS_TEMPOS)})')
print(f'Arquivo salvo: {SAIDA}')

Colunas: 46 -> 39 (descartadas: 11, novas: 4)
Arquivo salvo: csvs/07_dataset_refinado.csv


## 10. Constantes dentro da corrida: depende do modelo

A secao 10 de `06_Avaliacao_Colunas.ipynb` classifica `race_year`, `race_round`, `race_qualifying_format`, `race_direction`, `race_course_length`, `race_turns`, `circuit_id`, `circuit_type`, `layout_id`, `layout_effective` e `gp_id` como constantes dentro da corrida: nao ordenam os pilotos sozinhas, mas ainda podem ajudar em interacoes (arvore) ou na divisao temporal.

| Coluna | Decisao |
|---|---|
| `race_year` | Metadado, nao feature: usado so para a divisao temporal (`dataset_split`). Validacao (2025) e teste (2026) ficam fora da faixa do treino, e arvores nao extrapolam. |
| `race_round` | Mantido como feature: combina com o standing anterior (secao 4) — liderar na rodada 3 nao e o mesmo que liderar na rodada 20. |
| `circuit_type`, `race_direction` | Mantidos como feature: ajudam modelos de arvore (interacoes equipe x pista de rua). Descartar antes de um modelo linear. |
| `circuit_id` | Mantido (opcional): `driver_circuit_prior_*` (`feature_engineering.py`) ja traz a parte defasada util, mas ainda nao foi portado para este notebook. |
| `layout_id`, `gp_id`, `layout_effective` | Descartados: repetem `circuit_id`. |
| `race_qualifying_format` | Descartado: so marca a epoca. |
| — | Acrescenta `race_grid_size` (N de largantes da corrida, varia de 18 a 24) como feature, em vez de normalizar o alvo. |</cell_type>


In [16]:
METADADOS = METADADOS + ['race_year']  # divisao temporal, nao feature: 2025/2026 ficam fora da faixa do treino
print(f'Colunas de metadado (mantidas, fora do conjunto de features): {METADADOS}')

divisao = df[['race_id', 'driver_id', 'race_year']].copy()
divisao['dataset_split'] = 'train'
divisao.loc[divisao['race_year'].eq(2025), 'dataset_split'] = 'validation'
divisao.loc[divisao['race_year'].eq(2026), 'dataset_split'] = 'test'
divisao['race_grid_size'] = df.groupby('race_id')['driver_id'].transform('size')

NOVAS_CONSTANTES = ['dataset_split', 'race_grid_size']

assert divisao['race_grid_size'].between(18, 24).all()

resumo_split = divisao.groupby('dataset_split').agg(
    corridas=('race_id', 'nunique'), linhas=('driver_id', 'size'),
    primeiro_ano=('race_year', 'min'), ultimo_ano=('race_year', 'max'),
)
display(resumo_split)

Colunas de metadado (mantidas, fora do conjunto de features): ['race_id', 'race_date', 'race_year']


,corridas,linhas,primeiro_ano,ultimo_ano
dataset_split,,,,
test,8,176,2026,2026
train,379,7966,2003,2024
validation,16,320,2025,2025


In [17]:
DESCARTADAS_CONSTANTES = ['layout_id', 'gp_id', 'layout_effective', 'race_qualifying_format']
assert len(DESCARTADAS_CONSTANTES) == 4

antes = df_refinado.shape[1]
df_refinado = df_refinado.merge(
    divisao[['race_id', 'driver_id'] + NOVAS_CONSTANTES],
    on=['race_id', 'driver_id'], how='left', validate='one_to_one',
)
df_refinado = df_refinado.drop(columns=DESCARTADAS_CONSTANTES)

assert len(df_refinado) == len(df)
assert set(NOVAS_CONSTANTES) <= set(df_refinado.columns)
assert not set(DESCARTADAS_CONSTANTES) & set(df_refinado.columns)
assert set(METADADOS) <= set(df_refinado.columns)

df_refinado.to_csv(SAIDA, index=False)
print(f'Colunas: {antes} -> {df_refinado.shape[1]} (descartadas: {len(DESCARTADAS_CONSTANTES)}, novas: {len(NOVAS_CONSTANTES)})')
print(f'Arquivo salvo: {SAIDA}')

Colunas: 39 -> 37 (descartadas: 4, novas: 2)
Arquivo salvo: csvs/07_dataset_refinado.csv
